# Supplementary Table S12: Cross-validation of the idea diffusion model

This notebook evaluates Model (4) of Table 1 (weighted logistic regression for the fraction of ideas diffused from an origin country to a destination country) with 5-fold cross-validation. Performance is McFadden's pseudo-$R^2$ on the held-out data.

It reads the processed data in the data folder (`path2data`) and can be run on its own.

In [1]:
import os
import json
import gzip
import itertools
from itertools import chain
import pandas as pd
import numpy as np
import networkx as nx

import scipy.sparse as spsparse
import scipy.stats as spstats

import statsmodels.api as sm

import matplotlib.pylab as plt
import matplotlib.gridspec as gridspec
import matplotlib.transforms as mtransforms
import seaborn as sns


# set this path to where the OpenAlex database is stored
path2openalex = '/Users/hgt6rn/Documents/DataSets/OpenAlex'

# set this path to where the processed data (Source Data) is stored:
path2sourcedata = "Source Data"
# when the notebook is run from the code folder, the Source Data folder is one level up
if not os.path.exists(path2sourcedata):
    path2sourcedata = os.path.join("..", "Source Data")

# the processed input data are in the data folder, next to the Source Data folder
path2data = os.path.join(os.path.dirname(os.path.abspath(path2sourcedata)), 'data')

from sklearn.model_selection import KFold

## Supplementary Table S12: 5-fold cross-validation of Model (4)

In [2]:
panel_data = pd.read_csv(os.path.join(path2data, 'diffusion_panel_data_1990_2017.csv.gz'))

# the variables of Model (4) in Table 1
model_features = ['GDP_PCAP_o', 'GDP_PCAP_d', 'NumPub_o', 'NumPub_d', 'NormNcites', 'FieldDistance', 'dist', 'comlang_off',
                  'Pedge', 'Nedge', 'SameComm', 'pr_o', 'pr_d']

X = panel_data[model_features + ['Norigin', 'Fideas']].dropna().copy()
y = X['Fideas']
freq_weights = X['Norigin']
X = sm.add_constant(X[model_features])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

kfold = []
for fold, (train_index, test_index) in enumerate(kf.split(X)):
    X_train, X_test = X.iloc[train_index].copy(), X.iloc[test_index].copy()
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    weights_train, weights_test = freq_weights.iloc[train_index], freq_weights.iloc[test_index]

    # standardize with the training mean and standard deviation
    for c in model_features:
        if X_train[c].nunique() > 2:
            mean, std = X_train[c].mean(), X_train[c].std()
            X_train[c] = (X_train[c] - mean)/std
            X_test[c] = (X_test[c] - mean)/std

    model = sm.GLM(y_train, X_train, family=sm.families.Binomial(sm.families.links.logit()), freq_weights=weights_train).fit()

    # McFadden's pseudo R2 on the test set; the null model predicts the weighted mean of the training set
    pred = np.clip(model.predict(X_test), 1e-10, 1 - 1e-10)
    loglike = np.sum(weights_test*(y_test*np.log(pred) + (1 - y_test)*np.log(1 - pred)))
    mean_y = np.clip(np.average(y_train, weights=weights_train), 1e-10, 1 - 1e-10)
    loglike_null = np.sum(weights_test*(y_test*np.log(mean_y) + (1 - y_test)*np.log(1 - mean_y)))

    kfold.append([fold + 1, len(train_index), len(test_index), model.pseudo_rsquared(kind='mcf'), 1 - loglike/loglike_null])

kfold = pd.DataFrame(kfold, columns=['Fold', 'Training observations', 'Test observations', 'Pseudo R2 (training)',
                                     'Pseudo R2 (test)'])
print(kfold.round(4).to_string(index=False))
print('Mean {:.4f}, standard deviation {:.4f}'.format(kfold['Pseudo R2 (test)'].mean(), kfold['Pseudo R2 (test)'].std()))

/Users/psp2nq/anaconda3/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The logit link alias is deprecated. Use Logit instead. The logit link alias will be removed after the 0.15.0 release.
  warnings.warn(
/Users/psp2nq/anaconda3/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The logit link alias is deprecated. Use Logit instead. The logit link alias will be removed after the 0.15.0 release.
  warnings.warn(
/Users/psp2nq/anaconda3/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The logit link alias is deprecated. Use Logit instead. The logit link alias will be removed after the 0.15.0 release.
  warnings.warn(
/Users/psp2nq/anaconda3/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The logit link alias is deprecated. Use Logit instead. The logit link alias will be removed after the 0.15.0 release.
  warnings.warn(


 Fold  Training observations  Test observations  Pseudo R2 (training)  Pseudo R2 (test)
    1                   9625               2407                0.4158            0.3880
    2                   9625               2407                0.4265            0.3326
    3                   9626               2406                0.4313            0.3176
    4                   9626               2406                0.4364            0.3011
    5                   9626               2406                0.4143            0.3750
Mean 0.3429, standard deviation 0.0373


/Users/psp2nq/anaconda3/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The logit link alias is deprecated. Use Logit instead. The logit link alias will be removed after the 0.15.0 release.
  warnings.warn(


## Source Data for Supplementary Table 12

The values reported in the table, written to `Source Data/SourceData_12_TableS12.xlsx`.

In [3]:
# the Source Data helpers (sourcedatautils.py, in the code folder like vizutils.py)
from sourcedatautils import write_source_data

summary = pd.DataFrame({'Statistic':['Mean', 'Standard deviation'],
                        'Pseudo R2 (test)':[kfold['Pseudo R2 (test)'].mean(), kfold['Pseudo R2 (test)'].std()]})
write_source_data('SourceData_12_TableS12.xlsx',
                  'Supplementary Table 12: Performance of the 5-fold cross-validation of Model (4) for idea diffusion.',
                  ('Model (4) of Table 1: weighted logistic regression for the fraction of ideas diffused from an origin '
                   'country to a destination country (weights: number of ideas originating from the origin country). '
                   'Continuous variables are standardized with the mean and standard deviation of the training set. '
                   "Performance is McFadden's pseudo R2; on the test set, the null model predicts the weighted mean of the "
                   'training set. Folds are random (random_state = 42).'),
                  [('Folds', 'Pseudo R2 of each fold.', kfold), ('Summary', 'Mean and standard deviation over the folds.', summary)])

wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/SourceData_12_TableS12.xlsx
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/README.txt
